In [1]:
import yfinance as yf

stk_data = yf.download("ITC.NS",
                       start="2021-06-01",
                       end="2026-02-22")

# Remove MultiIndex if present
if isinstance(stk_data.columns, tuple) or hasattr(stk_data.columns, 'levels'):
    stk_data.columns = stk_data.columns.get_level_values(0)

stk_data = stk_data[['Open', 'High', 'Low', 'Close', 'Volume']]

print(stk_data.head())

[*********************100%***********************]  1 of 1 completed

Price             Open        High         Low       Close     Volume
Date                                                                 
2021-06-01  171.372448  172.473006  168.424528  169.210632   40247686
2021-06-02  165.476600  167.009519  163.982983  164.297424  100377464
2021-06-03  165.673099  165.673099  164.061564  164.336716   48454441
2021-06-04  164.690473  164.847693  163.511310  164.100891   31286264
2021-06-07  165.397955  166.970176  165.240734  166.223373   32643869


In [3]:
stk_data=stk_data[["Open","High","Low","Close"]]
#stk_data.to_csv("Tatacoffee13_21.csv")

In [4]:
stk_data

Price,Open,High,Low,Close
Date,,,,
2021-06-01,171.372448,172.473006,168.424528,169.210632
2021-06-02,165.476600,167.009519,163.982983,164.297424
2021-06-03,165.673099,165.673099,164.061564,164.336716
2021-06-04,164.690473,164.847693,163.511310,164.100891
2021-06-07,165.397955,166.970176,165.240734,166.223373
...,...,...,...,...
2026-02-16,313.750000,318.250000,313.149994,317.950012
2026-02-17,318.450012,328.049988,318.149994,325.450012
2026-02-18,326.000000,332.950012,324.799988,332.450012


In [5]:
from sklearn.preprocessing import MinMaxScaler
Ms = MinMaxScaler()
data1= Ms.fit_transform(stk_data)
print("Len:",data1.shape)

Len: (1172, 4)


In [14]:
import pandas as pd
data1=pd.DataFrame(data1,columns=["Open","High","Low","Close"])

In [15]:
data1

,Open,High,Low,Close
0,0.026040,0.028042,0.019012,0.020873
1,0.007104,0.010876,0.004768,0.005222
2,0.007735,0.006678,0.005020,0.005347
3,0.004579,0.004084,0.003255,0.004596
4,0.006852,0.010753,0.008802,0.011357
...,...,...,...,...
1167,0.483310,0.486052,0.483154,0.494670
1168,0.498405,0.516842,0.499189,0.518560
1169,0.522653,0.532238,0.520516,0.540858
1170,0.548507,0.536322,0.521157,0.520312


In [16]:
training_size = round(len(data1 ) * 0.80)
print(training_size)
X_train=data1[:training_size]
X_test=data1[training_size:]
print("X_train length:",X_train.shape)
print("X_test length:",X_test.shape)
y_train=data1[:training_size]
y_test=data1[training_size:]
print("y_train length:",y_train.shape)
print("y_test length:",y_test.shape)

938
X_train length: (938, 4)
X_test length: (234, 4)
y_train length: (938, 4)
y_test length: (234, 4)


In [17]:
import warnings
warnings.filterwarnings("ignore")

In [18]:
performance={"Model":[],"RMSE":[],"MaPe":[],"Lag":[],"Test":[]}

In [22]:
def cominbation(dataset,listt):
    print(listt)
    datasetTwo=dataset[listt]
    test_obs = 28
    train =datasetTwo[:-test_obs]
    test = datasetTwo[-test_obs:]
    from statsmodels.tsa.api import VAR
    for i in [1,2,3,4,5,6,7,8,9,10]:
        model = VAR(train)
        results = model.fit(i)
        print('Order =', i)
        print('AIC: ', results.aic)
        print('BIC: ', results.bic)
        print()
    x = model.select_order(maxlags=12)
    order=x.selected_orders["aic"]
    result = model.fit(order)
    #result.summary()
    lagged_Values = train.values[-order:]
    pred = result.forecast(y=lagged_Values,steps=28) 
    preds=pd.DataFrame(pred,columns=listt)
    preds.to_csv("varforecasted_{}.csv".format(test_obs))
    from sklearn.metrics import mean_squared_error
    rmse= round(mean_squared_error(test,pred))
    from sklearn.metrics import mean_absolute_percentage_error
    mape=mean_absolute_percentage_error(test,pred)
    performance["Model"].append(listt)
    performance["RMSE"].append(rmse)
    performance["MaPe"].append(mape)
    performance["Lag"].append(order)
    performance["Test"].append(test_obs)
    perf=pd.DataFrame(performance)
    return perf,result,pred

In [23]:
listt=["Close","High","Open","Low"]
#listt=["AQI_calculated","PM10","PM2.5","NOx","NO2","NO","NH3","SO2","CO",'year']

In [24]:
perf,result,pred=cominbation(data1,listt)

['Close', 'High', 'Open', 'Low']
Order = 1
AIC:  -39.92484061877666
BIC:  -39.83662694136993

Order = 2
AIC:  -39.95302536523857
BIC:  -39.79412929689551

Order = 3
AIC:  -39.94923276884919
BIC:  -39.719554996550485

Order = 4
AIC:  -39.93731637472975
BIC:  -39.63675735647148

Order = 5
AIC:  -39.91859116720836
BIC:  -39.54705113127371

Order = 6
AIC:  -39.901499300961554
BIC:  -39.458878245189474

Order = 7
AIC:  -39.88526876018335
BIC:  -39.371466451234305

Order = 8
AIC:  -39.87214859955867
BIC:  -39.287064572177414

Order = 9
AIC:  -39.86238490586535
BIC:  -39.20591846214069

Order = 10
AIC:  -39.85068325992532
BIC:  -39.12273346854685



In [25]:
data1

,Open,High,Low,Close
0,0.026040,0.028042,0.019012,0.020873
1,0.007104,0.010876,0.004768,0.005222
2,0.007735,0.006678,0.005020,0.005347
3,0.004579,0.004084,0.003255,0.004596
4,0.006852,0.010753,0.008802,0.011357
...,...,...,...,...
1167,0.483310,0.486052,0.483154,0.494670
1168,0.498405,0.516842,0.499189,0.518560
1169,0.522653,0.532238,0.520516,0.540858
1170,0.548507,0.536322,0.521157,0.520312


In [26]:
perf

,Model,RMSE,MaPe,Lag,Test
0,"[Close, High, Open, Low]",0,0.091331,2,28
